In [1]:
import os
import math
import copy
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

loadPath = '/home/sz4544/EEG-motor-imagery-main/project/'
model_path = os.path.join(loadPath, "models", "latent_diffusion.pt")

f = h5py.File(os.path.join(loadPath, "train1800_latent.h5"), "r")
z_train = f["latent"][:]
y_train = f["tasks"][:]
f.close()

y_train = y_train.astype(np.int64)
if np.array_equal(np.unique(y_train), np.array([1,2,3,4])):
    y_train = y_train - 1

print("z_train:", z_train.shape)
print("labels:", np.unique(y_train, return_counts=True))

latent_dim = z_train.shape[1]

class LatentDataset(Dataset):
    def __init__(self, z, y):
        self.z = torch.tensor(z, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.z)
    def __getitem__(self, idx):
        return self.z[idx], self.y[idx]

dataset = LatentDataset(z_train, y_train)
loader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)

class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half_dim = self.dim // 2
        emb = math.log(10000) / max(half_dim - 1, 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None] * emb[None, :]
        return torch.cat((emb.sin(), emb.cos()), dim=1)

class LatentDiffusionMLP(nn.Module):
    def __init__(self, latent_dim=128, n_classes=4, hidden=256, time_dim=128):
        super().__init__()
        self.time_emb = SinusoidalPosEmb(time_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden)
        )
        self.class_emb = nn.Embedding(n_classes, hidden)

        self.net = nn.Sequential(
            nn.Linear(latent_dim + hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, latent_dim)
        )

    def forward(self, z, t, y):
        t_emb = self.time_mlp(self.time_emb(t))
        c_emb = self.class_emb(y)
        cond = t_emb + c_emb
        inp = torch.cat([z, cond], dim=1)
        return self.net(inp)

T = 500
betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

def q_sample(z0, t, noise):
    a = alphas_cumprod[t].view(-1, 1)
    return torch.sqrt(a) * z0 + torch.sqrt(1 - a) * noise

class EMA:
    def __init__(self, beta=0.999):
        self.beta = beta
    def update_model_average(self, ema_model, model):
        with torch.no_grad():
            for ema_param, param in zip(ema_model.parameters(), model.parameters()):
                ema_param.data = self.beta * ema_param.data + (1 - self.beta) * param.data

model = LatentDiffusionMLP(latent_dim=latent_dim).to(device)
ema_model = copy.deepcopy(model).to(device)
ema_model.eval()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
ema = EMA(beta=0.999)

epochs = 200
model.train()

for epoch in range(epochs):
    losses = []
    for zb, yb in loader:
        zb = zb.to(device)
        yb = yb.to(device)

        t = torch.randint(0, T, (zb.size(0),), device=device)
        noise = torch.randn_like(zb)
        zt = q_sample(zb, t, noise)

        pred = model(zt, t.float(), yb)
        loss = F.mse_loss(pred, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        ema.update_model_average(ema_model, model)

        losses.append(loss.item())

    print(f"epoch {epoch+1}/{epochs} loss={np.mean(losses):.6f}")

torch.save(ema_model.state_dict(), model_path)
print("saved:", model_path)

cuda
z_train: (1800, 128)
labels: (array([0, 1, 2, 3]), array([450, 450, 450, 450]))
epoch 1/200 loss=1.571325
epoch 2/200 loss=1.010159
epoch 3/200 loss=1.003038
epoch 4/200 loss=0.998262
epoch 5/200 loss=1.002931
epoch 6/200 loss=1.003210
epoch 7/200 loss=0.996915
epoch 8/200 loss=0.996718
epoch 9/200 loss=1.003079
epoch 10/200 loss=0.999096
epoch 11/200 loss=1.002314
epoch 12/200 loss=0.995791
epoch 13/200 loss=0.997774
epoch 14/200 loss=1.003275
epoch 15/200 loss=0.991416
epoch 16/200 loss=0.998487
epoch 17/200 loss=0.997722
epoch 18/200 loss=1.000686
epoch 19/200 loss=1.003435
epoch 20/200 loss=1.001358
epoch 21/200 loss=0.999966
epoch 22/200 loss=1.002231
epoch 23/200 loss=0.995345
epoch 24/200 loss=0.998176
epoch 25/200 loss=0.999460
epoch 26/200 loss=0.999102
epoch 27/200 loss=0.992415
epoch 28/200 loss=1.003275
epoch 29/200 loss=1.003105
epoch 30/200 loss=0.999298
epoch 31/200 loss=1.005680
epoch 32/200 loss=1.000069
epoch 33/200 loss=0.999234
epoch 34/200 loss=1.005693
epoch 